<a href="https://colab.research.google.com/github/Sav200512/sp500-rfm-segmentation-cw/blob/main/Task_1_and_2_cw_(20232749_W2151870).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1.1

In [7]:
import os, json, platform
from io import StringIO
import requests
import numpy as np
import pandas as pd
import yfinance as yf

In [8]:
SEED = 42
START, END = "2021-01-01", "2025-01-01"   # END is exclusive
BENCH = "^GSPC"

for folder in ["data/raw", "data/processed", "figures"]:
    os.makedirs(folder, exist_ok=True)

print("pandas", pd.__version__, "| numpy", np.__version__, "| yfinance", yf.__version__)

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


In [9]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
sp500 = pd.read_html(StringIO(html))[0]

sp500["Yahoo_Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500["Retrieved_On"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M UTC")

sp500_out = sp500[["Symbol", "Yahoo_Symbol", "Security", "GICS Sector", "Retrieved_On"]]
sp500_out.to_csv("data/raw/sp500_tickers.csv", index=False)

print("Tickers:", len(sp500_out), "| retrieved:", sp500_out["Retrieved_On"].iloc[0])
sp500_out.head()

Tickers: 503 | retrieved: 2026-11-11 04:12 UTC


,Symbol,Yahoo_Symbol,Security,GICS Sector,Retrieved_On
0,MMM,MMM,3M,Industrials,2026-11-11 04:12 UTC
1,AOS,AOS,A. O. Smith,Industrials,2026-11-11 04:12 UTC
2,ABT,ABT,Abbott Laboratories,Health Care,2026-11-11 04:12 UTC
3,ABBV,ABBV,AbbVie,Health Care,2026-11-11 04:12 UTC
4,ACN,ACN,Accenture,Information Technology,2026-11-11 04:12 UTC


In [10]:
tickers = sp500["Yahoo_Symbol"].tolist()
frames = []

for i in range(0, len(tickers), 50):
    batch = tickers[i:i + 50]
    d = yf.download(batch, start=START, end=END, auto_adjust=False,
                    progress=False, threads=True)
    frames.append(d)
    print(f"Batch {i // 50 + 1} done ({len(batch)} tickers)")

raw = pd.concat(frames, axis=1)

close = raw["Close"]
adj_close = raw["Adj Close"]
volume = raw["Volume"]

# Benchmark
bench = yf.download(BENCH, start=START, end=END, auto_adjust=False, progress=False)
if isinstance(bench.columns, pd.MultiIndex):
    bench.columns = bench.columns.get_level_values(0)

Batch 1 done (50 tickers)
Batch 2 done (50 tickers)
Batch 3 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 4 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 5 done (50 tickers)
Batch 6 done (50 tickers)
Batch 7 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 8 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 9 done (50 tickers)


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['VYLR']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


Batch 10 done (50 tickers)
Batch 11 done (3 tickers)


In [11]:
close.to_csv("data/raw/close_daily.csv.gz")
adj_close.to_csv("data/raw/adj_close_daily.csv.gz")
volume.to_csv("data/raw/volume_daily.csv.gz")
bench.to_csv("data/raw/gspc_daily.csv")

run_log = {
    "download_time_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "start": START, "end_exclusive": END, "benchmark": BENCH,
    "yfinance": yf.__version__, "pandas": pd.__version__,
    "python": platform.python_version(),
    "tickers_requested": len(tickers),
}
json.dump(run_log, open("data/raw/run_log.json", "w"), indent=2)

In [12]:
no_data = [t for t in tickers if t not in adj_close.columns or adj_close[t].isna().all()]

print("Requested:", len(tickers))
print("Returned with some data:", len(tickers) - len(no_data))
print("No data:", no_data)
print("Shares: first date", adj_close.index.min().date(), "| last date", adj_close.index.max().date())
print("Benchmark: first date", bench.index.min().date(), "| last date", bench.index.max().date(), "| rows", len(bench))
print("Share rows:", len(adj_close), "| shape close/adj/vol:", close.shape, adj_close.shape, volume.shape)

Requested: 503
Returned with some data: 498
No data: ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']
Shares: first date 2021-01-04 | last date 2024-12-31
Benchmark: first date 2021-01-04 | last date 2024-12-31 | rows 1005
Share rows: 1005 | shape close/adj/vol: (1005, 503) (1005, 503) (1005, 503)
